# MATH 70116, Deep Learning
## Autumn 2026
## Lecturer: Tom Hadfield
# Explicit backpropagation example

### $z^{i+1} = W^{i+1} a^i + b^{i+1}$     $\quad \quad i = 0,1,2$
### Layerwise activations $a^i$:
### $a^0 = x \quad \quad$ initial input
### $a^i = \sigma(z^i)$ $\quad \quad i = 1,2$, with $\sigma = \mathrm{ReLU}$ here
### $a^3 = z^3$, output
### For a data point $(x,y)$, loss function $L(a^3,y) = (a^3 - y)**2$ (MSE loss)
## Backprop equations:
### Define the error of neuron $j$ in layer $i$ as $\delta_j^i = {\frac{\partial L}{\partial z_j^i}}$
### For the output layer ($i = n = 3$ here) : $\quad \delta_j^n = {\frac{\partial L}{\partial z_j^n}} \sigma' (z_j^n)$
### For $i < n$ : $\quad \delta_j^i = ((W^{i+1})^t \delta^{i+1} )_j \sigma' (z_j^i) = \sum_k \delta_k^{i+1} W^{i+1}_{kj} \sigma' (z_j^i)$
### Biases : $\quad {\frac{\partial L}{\partial b_j^i}} = \delta_j^i$
### Weights : $\quad {\frac{\partial L}{\partial W_{jk}^i}} = a_k^{i-1} \delta_j^i$

In [1]:
import torch
import torch.nn as nn
import numpy as np

In [2]:
torch.manual_seed(0)

f = nn.Sequential(
    nn.Linear(2, 3),
    nn.ReLU(),
    nn.Linear(3, 2),
    nn.ReLU(),
    nn.Linear(2, 1)
)

# convert to float64 rather than float32 precision
# to be compatible with tensors defined by numpy arrays below
f = f.float()      

In [3]:
# define weights and biases for the linear layers manually, as numpy arrays

W1 = np.array([[1., 0.],[-1., -1.],[2., 0.]])
b1 = np.array([1., 0., -1.])

W2 = np.array([[1., 0., 2.],[-1., -1., 0.]])
b2 = np.array([0.5, -0.5])

W3 = np.array([[3., -2.]])
b3 = np.array([0.1])

In [4]:
# copy them into the network

with torch.no_grad():
    f[0].weight.copy_(torch.from_numpy(W1).float())
    f[0].bias.copy_(torch.from_numpy(b1).float())
    f[2].weight.copy_(torch.from_numpy(W2).float())
    f[2].bias.copy_(torch.from_numpy(b2).float())
    f[4].weight.copy_(torch.from_numpy(W3).float())
    f[4].bias.copy_(torch.from_numpy(b3).float())

In [5]:
print(f[0].weight.dtype)
print(f[0].bias.dtype)

torch.float32
torch.float32


In [6]:
linear1, relu1, linear2, relu2, linear3 = f

In [7]:
# Single input/target for this example

# input
x = np.array([1., -1.])
print(type(x), type(x[0]))
a0 = torch.from_numpy(x).float()
print("a0.dtype = ", a0.dtype)

# regression target
y_true = torch.tensor([[10.0]]) 

<class 'numpy.ndarray'> <class 'numpy.float64'>
a0.dtype =  torch.float32


In [8]:
# define ReLU

# apply ReLU to an array by vectorisation
def my_relu0(x):
  return max(x,0.)

# Apply the function using np.vectorize
my_relu = np.vectorize(my_relu0)

In [9]:
# quick check
x = np.random.uniform(-1,1,(2,3))
print(x)
print(my_relu(x))

[[ 0.83524538  0.62885053 -0.74017398]
 [ 0.0059475   0.09848886 -0.73797145]]
[[0.83524538 0.62885053 0.        ]
 [0.0059475  0.09848886 0.        ]]


## Forward pass (explicit, layer by layer)

### $z^1 = W^1 a^0 + b^1$

In [10]:
z1_np = W1 @ a0.numpy() + b1
print("z1 = W1 a0 + b1 = \n", W1, a0.numpy(), " + ", b1, " = ", z1_np)

z1 = linear1(a0)
print("\n","z1 = ", z1)

z1 = W1 a0 + b1 = 
 [[ 1.  0.]
 [-1. -1.]
 [ 2.  0.]] [ 1. -1.]  +  [ 1.  0. -1.]  =  [2. 0. 1.]

 z1 =  tensor([2., 0., 1.], grad_fn=<ViewBackward0>)


### $a^1 = \mathrm{ReLU}(z^1)$

In [11]:
a1_np = my_relu(z1_np)
print("a1 = ReLU(z1) = ", a1_np)

a1 = relu1(z1)
print("\n","a1 = ", a1)

a1 = ReLU(z1) =  [2. 0. 1.]

 a1 =  tensor([2., 0., 1.], grad_fn=<ReluBackward0>)


### $z^2 = W^2 a^1 + b^2$

In [12]:
z2_np = W2 @ a1.detach().numpy() + b2
print("z2 = W2 a1 + b2 = \n", W2, a1.detach().numpy(), " + ", b2, " = ", z2_np)

z2 = linear2(a1)
print("\n","z2 = ", z2)

z2 = W2 a1 + b2 = 
 [[ 1.  0.  2.]
 [-1. -1.  0.]] [2. 0. 1.]  +  [ 0.5 -0.5]  =  [ 4.5 -2.5]

 z2 =  tensor([ 4.5000, -2.5000], grad_fn=<ViewBackward0>)


### $a^2 = \mathrm{ReLU}(z^2)$

In [13]:
a2_np = my_relu(z2_np)
print("a2 = ReLU(z2) = ", a2_np)

a2 = relu2(z2)
print("\n","a2 = ", a2)

a2 = ReLU(z2) =  [4.5 0. ]

 a2 =  tensor([4.5000, 0.0000], grad_fn=<ReluBackward0>)


### $y_{pred} = z^3 = W^3 a^2 + b^3$

In [14]:
y_pred_np = W3 @ a2.detach().numpy() + b3
print("y_pred = z3 = W3 a2 + b3 = \n", W3, a2.detach().numpy(), " + ", b3, " = ", y_pred_np)

y_pred = linear3(a2)
print("\n","y_pred = ", y_pred)

y_pred = z3 = W3 a2 + b3 = 
 [[ 3. -2.]] [4.5 0. ]  +  [0.1]  =  [13.6]

 y_pred =  tensor([13.6000], grad_fn=<ViewBackward0>)


## MSE loss (manual, matching nn.MSELoss())

In [15]:
loss = ((y_pred - y_true) ** 2).mean()
print("loss = ", loss.item())

loss =  12.960002899169922


## Backward pass (explicit, via the chain rule)

In [16]:
# dL/dy_pred : 2*(y_pred - y_true) / n_elements  (from the .mean())
n = y_true.numel()
print("n = ", n)
dL_dypred = 2 * (y_pred - y_true) / n    # shape (1, 1)
print("dL_dypred = ", dL_dypred)

n =  1
dL_dypred =  tensor([[7.2000]], grad_fn=<DivBackward0>)


In [17]:
# No activation on the output layer, so dL/dz3 = dL/dy_pred
dL_dz3 = dL_dypred
print("dL_dz3 = ", dL_dz3)

dL_dz3 =  tensor([[7.2000]], grad_fn=<DivBackward0>)


In [18]:
# Gradients for linear3: z3 = a2 @ W3^T + b3
#print("dL_dz3.shape = ", dL_dz3.shape)
dL_dW3 = dL_dz3.t() @ a2.reshape(1,-1)
dL_db3 = dL_dz3.sum(dim=0)
print("dL_dW3 = ", dL_dW3)
print("dL_db3 = ", dL_db3)

dL_dW3 =  tensor([[32.4000,  0.0000]], grad_fn=<MmBackward0>)
dL_db3 =  tensor([7.2000], grad_fn=<SumBackward1>)


In [19]:
# Backprop through linear3 to get dL/da2
dL_da2 = dL_dz3 @ linear3.weight
print("dL_da2 = ", dL_da2)

dL_da2 =  tensor([[ 21.6000, -14.4000]], grad_fn=<MmBackward0>)


In [20]:
# Backprop through relu2
print("z2 = ", z2)
drelu2 = (z2 > 0).float()
print("drelu2 = ", drelu2)
dL_dz2 = dL_da2 * drelu2
print("dL_dz2 = ", dL_dz2)

z2 =  tensor([ 4.5000, -2.5000], grad_fn=<ViewBackward0>)
drelu2 =  tensor([1., 0.])
dL_dz2 =  tensor([[21.6000, -0.0000]], grad_fn=<MulBackward0>)


In [21]:
# Gradients for linear2
dL_dW2 = dL_dz2.t() @ a1.reshape(1,-1)
dL_db2 = dL_dz2.sum(dim=0)
print("dL_dW2 = ", dL_dW2)
print("dL_db2 = ", dL_db2)

dL_dW2 =  tensor([[43.2000,  0.0000, 21.6000],
        [-0.0000, -0.0000, -0.0000]], grad_fn=<MmBackward0>)
dL_db2 =  tensor([21.6000,  0.0000], grad_fn=<SumBackward1>)


In [22]:
# Backprop through linear2 to get dL/da1
dL_da1 = dL_dz2 @ linear2.weight
print("dL_da1 = ", dL_da1)

dL_da1 =  tensor([[21.6000,  0.0000, 43.2000]], grad_fn=<MmBackward0>)


In [23]:
# Backprop through relu1
print("z1 = ", z1)
drelu1 = (z1 > 0).float()
print("drelu1 = ", drelu1)
dL_dz1 = dL_da1 * drelu1
print("dL_dz1 = ", dL_dz1)

z1 =  tensor([2., 0., 1.], grad_fn=<ViewBackward0>)
drelu1 =  tensor([1., 0., 1.])
dL_dz1 =  tensor([[21.6000,  0.0000, 43.2000]], grad_fn=<MulBackward0>)


In [24]:
# Gradients for linear1
dL_dW1 = dL_dz1.t() @ a0.reshape(1,-1)
dL_db1 = dL_dz1.sum(dim=0)
print("dL_dW1 = ", dL_dW1)
print("dL_db1 = ", dL_db1)

dL_dW1 =  tensor([[ 21.6000, -21.6000],
        [  0.0000,  -0.0000],
        [ 43.2000, -43.2000]], grad_fn=<MmBackward0>)
dL_db1 =  tensor([21.6000,  0.0000, 43.2000], grad_fn=<SumBackward1>)


In [25]:
print("\nManual gradients:")
print("dL/dW1 =", dL_dW1)
print("dL/db1 =", dL_db1)
print("dL/dW2 =", dL_dW2)
print("dL/db2 =", dL_db2)
print("dL/dW3 =", dL_dW3)
print("dL/db3 =", dL_db3)


Manual gradients:
dL/dW1 = tensor([[ 21.6000, -21.6000],
        [  0.0000,  -0.0000],
        [ 43.2000, -43.2000]], grad_fn=<MmBackward0>)
dL/db1 = tensor([21.6000,  0.0000, 43.2000], grad_fn=<SumBackward1>)
dL/dW2 = tensor([[43.2000,  0.0000, 21.6000],
        [-0.0000, -0.0000, -0.0000]], grad_fn=<MmBackward0>)
dL/db2 = tensor([21.6000,  0.0000], grad_fn=<SumBackward1>)
dL/dW3 = tensor([[32.4000,  0.0000]], grad_fn=<MmBackward0>)
dL/db3 = tensor([7.2000], grad_fn=<SumBackward1>)


## Verify against PyTorch's autograd

In [26]:
f.zero_grad()
loss.backward()

In [27]:
print(f[0].weight.grad)

tensor([[ 21.6000, -21.6000],
        [  0.0000,  -0.0000],
        [ 43.2000, -43.2000]])


In [28]:
print("\nAutograd gradients (should match manual ones above):")

print("\nManual dL/dW1 =", dL_dW1)
print("Autograd dL/dW1 =", f[0].weight.grad)
dL_dW1_error = (torch.sum(dL_dW1 - f[0].weight.grad)**2).item()
print("dL_dW1_error =", dL_dW1_error)

print("\nManual dL/db1 =", dL_db1)
print("Autograd dL/db1 =", linear1.bias.grad)
dL_db1_error = (torch.sum(dL_db1 - linear1.bias.grad)**2).item()
print("dL_db1_error =", dL_db1_error)

print("\nManual dL/dW2 =", dL_dW2)
print("Autograd dL/dW2 =", linear2.weight.grad)
dL_dW2_error = (torch.sum(dL_dW2 - linear2.weight.grad)**2).item()
print("dL_dW2_error =", dL_dW2_error)

print("\nManual dL/db2 =", dL_db2)
print("Autograd dL/db2 =", linear2.bias.grad)
dL_db2_error = (torch.sum(dL_db2 - linear2.bias.grad)**2).item()
print("dL_db2_error =", dL_db2_error)

print("\nManual dL/dW3 =", dL_dW3)
print("Autograd dL/dW3 =", linear3.weight.grad)
dL_dW3_error = (torch.sum(dL_dW3 - linear3.weight.grad)**2).item()
print("dL_dW3_error =", dL_dW3_error)

print("\nManual dL/db3 =", dL_db3)
print("Autograd dL/db3 =", linear3.bias.grad)
dL_db3_error = (torch.sum(dL_db3 - linear3.bias.grad)**2).item()
print("dL_db3_error =", dL_db3_error)



Autograd gradients (should match manual ones above):

Manual dL/dW1 = tensor([[ 21.6000, -21.6000],
        [  0.0000,  -0.0000],
        [ 43.2000, -43.2000]], grad_fn=<MmBackward0>)
Autograd dL/dW1 = tensor([[ 21.6000, -21.6000],
        [  0.0000,  -0.0000],
        [ 43.2000, -43.2000]])
dL_dW1_error = 0.0

Manual dL/db1 = tensor([21.6000,  0.0000, 43.2000], grad_fn=<SumBackward1>)
Autograd dL/db1 = tensor([21.6000,  0.0000, 43.2000])
dL_db1_error = 0.0

Manual dL/dW2 = tensor([[43.2000,  0.0000, 21.6000],
        [-0.0000, -0.0000, -0.0000]], grad_fn=<MmBackward0>)
Autograd dL/dW2 = tensor([[43.2000,  0.0000, 21.6000],
        [ 0.0000,  0.0000,  0.0000]])
dL_dW2_error = 0.0

Manual dL/db2 = tensor([21.6000,  0.0000], grad_fn=<SumBackward1>)
Autograd dL/db2 = tensor([21.6000,  0.0000])
dL_db2_error = 0.0

Manual dL/dW3 = tensor([[32.4000,  0.0000]], grad_fn=<MmBackward0>)
Autograd dL/dW3 = tensor([[32.4000,  0.0000]])
dL_dW3_error = 0.0

Manual dL/db3 = tensor([7.2000], grad_fn=<

In [29]:
x = 1.0 * torch.tensor(np.arange(0,6)).reshape(2,3)
print(x)
print(x**2)
print(torch.sum(x**2).item())

tensor([[0., 1., 2.],
        [3., 4., 5.]])
tensor([[ 0.,  1.,  4.],
        [ 9., 16., 25.]])
55.0


### Perform one learning step, redo the forward pass, re-calculate the loss

In [30]:
lr = 0.001 # learning rate

W1 -= lr * dL_dW1.detach().numpy()
b1 -= lr * dL_db1.detach().numpy()

W2 -= lr * dL_dW2.detach().numpy()
b2 -= lr * dL_db2.detach().numpy()

W3 -= lr * dL_dW3.detach().numpy()
b3 -= lr * dL_db3.detach().numpy()

In [31]:
# copy the adjusted weights and biases back into the network

with torch.no_grad():
    f[0].weight.copy_(torch.from_numpy(W1).float())
    f[0].bias.copy_(torch.from_numpy(b1).float())
    f[2].weight.copy_(torch.from_numpy(W2).float())
    f[2].bias.copy_(torch.from_numpy(b2).float())
    f[4].weight.copy_(torch.from_numpy(W3).float())
    f[4].bias.copy_(torch.from_numpy(b3).float())

In [32]:
# same input data point as before:
x = np.array([1., -1.])
a0 = torch.from_numpy(x).float()
print("Input data : x = ", x)

Input data : x =  [ 1. -1.]


In [33]:
# Forward pass (no_grad since we're just doing inference)
with torch.no_grad():
    new_y_pred = f(a0)

# Convert back to numpy
new_y_pred_np = new_y_pred.numpy()

In [34]:
print("y_true = ", y_true.item())
print("learning rate = ", lr)

print("\nPrevious : y_pred = ", y_pred.item())
prev_loss = ((y_pred - y_true) ** 2).mean()
print("Previous loss = ", prev_loss.item())

print("\nNew : new_y_pred = ", new_y_pred.item())
new_loss = ((new_y_pred - y_true) ** 2).mean()
print("New loss = ", new_loss.item())

y_true =  10.0
learning rate =  0.001

Previous : y_pred =  13.600000381469727
Previous loss =  12.960002899169922

New : new_y_pred =  12.117511749267578
New loss =  4.483856201171875


## Exercises:
### 1. what happens as we increase the learning rate?
### 2. redo, for $N$ data points $x_1, ... , x_N$, with corresponding $y_i = g(x_i)$ for some function $g$ that you choose
### 3. redo, for classification rather than regression - so the labels are now 0,1 and the loss function is BCELoss (Binary Cross-Entropy) 